In [2]:
import pandas as pd

In [3]:
# Read the database
df = pd.read_csv("marketing_campaign_dataset.csv")

Lets check out the database, check its shape, column types, and verify for no repeat errors.

In [4]:
df.shape # rows, columns

(200000, 16)

In [5]:
df.dtypes

Campaign_ID           int64
Company                 str
Campaign_Type           str
Target_Audience         str
Duration                str
Channel_Used            str
Conversion_Rate     float64
Acquisition_Cost        str
ROI                 float64
Location                str
Language                str
Clicks                int64
Impressions           int64
Engagement_Score      int64
Customer_Segment        str
Date                    str
dtype: object

In [6]:
df.head()

,Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Location,Language,Clicks,Impressions,Engagement_Score,Customer_Segment,Date
0,1,Innovate Industries,Email,Men 18-24,30 days,Google Ads,0.04,"$16,174.00",6.29,Chicago,Spanish,506,1922,6,Health & Wellness,2021-01-01
1,2,NexGen Systems,Email,Women 35-44,60 days,Google Ads,0.12,"$11,566.00",5.61,New York,German,116,7523,7,Fashionistas,2021-01-02
2,3,Alpha Innovations,Influencer,Men 25-34,30 days,YouTube,0.07,"$10,200.00",7.18,Los Angeles,French,584,7698,1,Outdoor Adventurers,2021-01-03
3,4,DataTech Solutions,Display,All Ages,60 days,YouTube,0.11,"$12,724.00",5.55,Miami,Mandarin,217,1820,7,Health & Wellness,2021-01-04
4,5,NexGen Systems,Email,Men 25-34,15 days,YouTube,0.05,"$16,452.00",6.50,Los Angeles,Mandarin,379,4201,3,Health & Wellness,2021-01-05


In [7]:
# Need to change date to datetime
df['Date'] = pd.to_datetime(df['Date'])

In [8]:
# Now change Acquisition cost to a float, ingoring '$' and ','
df['Acquisition_Cost'] = (
    df['Acquisition_Cost']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

In [9]:
# Verify column types again
df.dtypes

Campaign_ID                  int64
Company                        str
Campaign_Type                  str
Target_Audience                str
Duration                       str
Channel_Used                   str
Conversion_Rate            float64
Acquisition_Cost           float64
ROI                        float64
Location                       str
Language                       str
Clicks                       int64
Impressions                  int64
Engagement_Score             int64
Customer_Segment               str
Date                datetime64[us]
dtype: object

Clean and validate the data

In [10]:
# Coutn missing values per column
df.isnull().sum()

# Count duplicates
df.duplicated().sum()

np.int64(0)

In [11]:
# Verify no repeats for channels & campain type
df['Channel_Used'].unique()

df['Campaign_Type'].unique()

<StringArray>
['Email', 'Influencer', 'Display', 'Search', 'Social Media']
Length: 5, dtype: str

Moving onto to calculating our data. We want to find Click through rate (CTR), Cost per click (CPC), Implied Revenue (since ROI given as a ration)

In [12]:
# CTR
df['CTR'] = df['Clicks'] / df['Impressions']
df.head()


,Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Location,Language,Clicks,Impressions,Engagement_Score,Customer_Segment,Date,CTR
0,1,Innovate Industries,Email,Men 18-24,30 days,Google Ads,0.04,16174.0,6.29,Chicago,Spanish,506,1922,6,Health & Wellness,2021-01-01,0.263267
1,2,NexGen Systems,Email,Women 35-44,60 days,Google Ads,0.12,11566.0,5.61,New York,German,116,7523,7,Fashionistas,2021-01-02,0.015419
2,3,Alpha Innovations,Influencer,Men 25-34,30 days,YouTube,0.07,10200.0,7.18,Los Angeles,French,584,7698,1,Outdoor Adventurers,2021-01-03,0.075864
3,4,DataTech Solutions,Display,All Ages,60 days,YouTube,0.11,12724.0,5.55,Miami,Mandarin,217,1820,7,Health & Wellness,2021-01-04,0.119231
4,5,NexGen Systems,Email,Men 25-34,15 days,YouTube,0.05,16452.0,6.50,Los Angeles,Mandarin,379,4201,3,Health & Wellness,2021-01-05,0.090217


In [13]:
# CPC
df['CPC'] = df['Acquisition_Cost'] / df['Clicks']
df.head()

,Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Location,Language,Clicks,Impressions,Engagement_Score,Customer_Segment,Date,CTR,CPC
0,1,Innovate Industries,Email,Men 18-24,30 days,Google Ads,0.04,16174.0,6.29,Chicago,Spanish,506,1922,6,Health & Wellness,2021-01-01,0.263267,31.964427
1,2,NexGen Systems,Email,Women 35-44,60 days,Google Ads,0.12,11566.0,5.61,New York,German,116,7523,7,Fashionistas,2021-01-02,0.015419,99.706897
2,3,Alpha Innovations,Influencer,Men 25-34,30 days,YouTube,0.07,10200.0,7.18,Los Angeles,French,584,7698,1,Outdoor Adventurers,2021-01-03,0.075864,17.465753
3,4,DataTech Solutions,Display,All Ages,60 days,YouTube,0.11,12724.0,5.55,Miami,Mandarin,217,1820,7,Health & Wellness,2021-01-04,0.119231,58.635945
4,5,NexGen Systems,Email,Men 25-34,15 days,YouTube,0.05,16452.0,6.50,Los Angeles,Mandarin,379,4201,3,Health & Wellness,2021-01-05,0.090217,43.408971


In [14]:
# Implied Revenue - Since not specified: 
# ** I am assuming dataset used standard ROI = (Revenue - Cost) / Cost **
df['Implied_Revenue'] = df['Acquisition_Cost'] * (1 + df['ROI'])
df['Implied_Revenue'].describe()


count    200000.000000
mean      75091.336411
std       34799.235244
min       15071.070000
25%       47728.085000
50%       68814.840000
75%       98146.365000
max      179438.360000
Name: Implied_Revenue, dtype: float64

In [15]:
# Ensure none of our calcuations left zeros in our new columns
print((df['Impressions'] == 0).sum())
print((df['Implied_Revenue'] == 0).sum())
print((df['Clicks'] == 0).sum())

0
0
0


In [19]:
# Make a new df with grouped info
channel_summary = df.groupby('Channel_Used').agg({
    'Acquisition_Cost': 'sum',
    'Implied_Revenue': 'sum',
    'Clicks': 'sum',
    'Impressions': 'sum',
    'ROI': 'mean',
    'Conversion_Rate': 'mean'

})
channel_summary.head()

,Acquisition_Cost,Implied_Revenue,Clicks,Impressions,ROI,Conversion_Rate
Channel_Used,,,,,,
Email,420874104.0,2.524644e+09,18493963,184801107,4.996487,0.080282
Facebook,410595258.0,2.474541e+09,18037947,180659428,5.018699,0.079992
Google Ads,418912314.0,2.516724e+09,18340807,185006879,5.003141,0.080183
Instagram,417124850.0,2.497101e+09,18316654,183738455,4.988706,0.079886
Website,416593500.0,2.504152e+09,18414628,183806353,5.014167,0.080183


In [21]:
# Channel leveled ROI
channel_summary['Recalculated_ROI'] = (
    (channel_summary['Implied_Revenue'] - channel_summary['Acquisition_Cost'])
    / channel_summary['Acquisition_Cost']
)
channel_summary.head()

,Acquisition_Cost,Implied_Revenue,Clicks,Impressions,ROI,Conversion_Rate,Recalculated_ROI
Channel_Used,,,,,,,
Email,420874104.0,2.524644e+09,18493963,184801107,4.996487,0.080282,4.998573
Facebook,410595258.0,2.474541e+09,18037947,180659428,5.018699,0.079992,5.026715
Google Ads,418912314.0,2.516724e+09,18340807,185006879,5.003141,0.080183,5.007758
Instagram,417124850.0,2.497101e+09,18316654,183738455,4.988706,0.079886,4.986460
Website,416593500.0,2.504152e+09,18414628,183806353,5.014167,0.080183,5.011021


In [25]:
# Segment level 
segment_summary = df.groupby(['Channel_Used', 'Customer_Segment']).agg({
    'Acquisition_Cost': 'sum',
    'Implied_Revenue': 'sum',
    'ROI': 'mean',
    'Conversion_Rate': 'mean'

})
segment_summary.sort_values('ROI', ascending=False)
segment_summary.head()

Acquisition_Cost  Implied_Revenue       ROI  \
Channel_Used Customer_Segment                                                   
Email        Fashionistas               84560315.0     5.067139e+08  4.987897   
             Foodies                    85312984.0     5.121162e+08  4.991678   
             Health & Wellness          82801606.0     4.986103e+08  5.021867   
             Outdoor Adventurers        83605440.0     4.984115e+08  4.970791   
             Tech Enthusiasts           84593759.0     5.087923e+08  5.010684   

                                  Conversion_Rate  
Channel_Used Customer_Segment                      
Email        Fashionistas                0.079736  
             Foodies                     0.080199  
             Health & Wellness           0.080436  
             Outdoor Adventurers         0.080605  
             Tech Enthusiasts            0.080442